# GP posterior as a distribution over functions (BoTorch + GPyTorch)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkarshp1161/Active-learning-in-microscopy/blob/main/notebooks/0_gp_posterior_function_sampling_botorch.ipynb)

This minimal notebook shows the *conceptual missing step*:

- `model.posterior(X_grid)` gives a **multivariate Gaussian** over values \(f(X_{grid})\).
- Sampling from that Gaussian gives **one possible curve** (a *function draw*).
- **Thompson sampling**: draw one curve, then pick its argmax as the next point.

> Requirements: `botorch`, `gpytorch`, `torch`, `matplotlib`.


In [ ]:
!pip install -q botorch==0.12.0
!pip install -q gpytorch==1.13

In [ ]:
import torch
import matplotlib.pyplot as plt

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

torch.manual_seed(0)


## 1) Toy 1D data

We'll learn an unknown function from a few noisy observations.


In [ ]:
# Training data (N x d) and observations (N x 1)
train_X = torch.tensor([[-2.0], [-0.5], [0.7], [2.2]])
true_f = lambda x: torch.sin(1.3 * x) + 0.15 * x
noise_std = 0.15

train_Y = true_f(train_X) + noise_std * torch.randn_like(train_X)
train_Y = train_Y  # shape (N, 1)

print(train_X.shape, train_Y.shape)


## 2) Fit a GP model (Exact GP)

BoTorch wraps a GPyTorch Exact GP. We'll use default priors and learn hyperparameters by maximizing the marginal likelihood.


In [ ]:
model = SingleTaskGP(train_X, train_Y)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)

model.eval()


## 3) Posterior at a grid: mean + uncertainty (what you usually plot)

This is still a distribution, but summarized by mean and variance.


In [ ]:
# Grid to evaluate posterior on
X_grid = torch.linspace(-3.0, 3.0, 300).unsqueeze(-1)

with torch.no_grad():
    posterior = model.posterior(X_grid)
    mean = posterior.mean.squeeze(-1)          # (300,)
    var = posterior.variance.squeeze(-1)       # (300,)
    sd = var.sqrt()

# Plot mean and 2-sigma band, plus data
plt.figure(figsize=(8, 4))
plt.plot(X_grid.squeeze(-1).numpy(), mean.numpy(), linewidth=2, label="posterior mean")
plt.fill_between(
    X_grid.squeeze(-1).numpy(),
    (mean - 2 * sd).numpy(),
    (mean + 2 * sd).numpy(),
    alpha=0.2,
    label="±2σ",
)
plt.scatter(train_X.squeeze(-1).numpy(), train_Y.squeeze(-1).numpy(), s=40, label="data")
plt.title("GP posterior summary: mean + uncertainty")
plt.legend()
plt.show()


## 4) Sampling a *function* from the posterior

Key idea:
\[ f(X_{grid}) \mid D \sim \mathcal{N}(\mu, \Sigma) \]

If we sample once from this multivariate Gaussian, we get one vector of values across the grid — that vector is one *function realization* (curve).

In BoTorch:
- `posterior.rsample(sample_shape=torch.Size([S]))` draws **S** correlated samples.


In [ ]:
with torch.no_grad():
    # Draw S posterior function samples evaluated on X_grid
    S = 8
    f_samples = posterior.rsample(sample_shape=torch.Size([S])).squeeze(-1)  # (S, 300)

plt.figure(figsize=(8, 4))
plt.scatter(train_X.squeeze(-1).numpy(), train_Y.squeeze(-1).numpy(), s=35, label="data")
for i in range(S):
    plt.plot(X_grid.squeeze(-1).numpy(), f_samples[i].numpy(), linewidth=1, alpha=0.9)
plt.title("Posterior samples = draws of a function (curves)")
plt.legend()
plt.show()

print("f_samples shape:", f_samples.shape)


## 5) Thompson sampling = posterior sampling + argmax decision

One Thompson step:
1. Sample one function from posterior.
2. Pick the maximizer of that sampled function as the next query point.

This is why TS can explore: different sampled functions can have different argmax locations.


In [ ]:
with torch.no_grad():
    # Sample ONE function over the grid
    f_one = posterior.rsample(sample_shape=torch.Size([1])).squeeze(-1).squeeze(0)  # (300,)
    idx = torch.argmax(f_one)
    x_next_ts = X_grid[idx].clone()

plt.figure(figsize=(8, 4))
plt.plot(X_grid.squeeze(-1).numpy(), f_one.numpy(), linewidth=2, label="one posterior function sample")
plt.scatter(train_X.squeeze(-1).numpy(), train_Y.squeeze(-1).numpy(), s=35, label="data")
plt.axvline(x_next_ts.item(), linestyle="--", linewidth=2, label=f"TS next x = {x_next_ts.item():.3f}")
plt.title("One Thompson sampling step (sample function → argmax)")
plt.legend()
plt.show()

x_next_ts


## 6) Compare with UCB in one line (optional)

UCB chooses:
\[ x_{next} = \arg\max_x \mu(x) + \beta\,\sigma(x) \]

Unlike TS, this is deterministic given \(\beta\).


In [ ]:
beta = 2.0
ucb = mean + beta * sd
idx_ucb = torch.argmax(ucb)
x_next_ucb = X_grid[idx_ucb].clone()

plt.figure(figsize=(8, 4))
plt.plot(X_grid.squeeze(-1).numpy(), mean.numpy(), linewidth=2, label="posterior mean")
plt.plot(X_grid.squeeze(-1).numpy(), ucb.numpy(), linewidth=2, label=f"UCB (beta={beta})")
plt.scatter(train_X.squeeze(-1).numpy(), train_Y.squeeze(-1).numpy(), s=35, label="data")
plt.axvline(x_next_ucb.item(), linestyle="--", linewidth=2, label=f"UCB next x = {x_next_ucb.item():.3f}")
plt.title("UCB decision (deterministic given beta)")
plt.legend()
plt.show()

x_next_ucb
